### Take best ensemble run parameter file and sample a smaller no. of parameters with smaller ranges

In [1]:
from scipy.stats import qmc
import numpy as np

import csv
import pandas as pd
import os
import netCDF4 as nc4
import sys
import shutil
from tempfile import TemporaryFile                                                                                                                                 
import argparse                                                                                                                                                                                                                                                                                                       
import tempfile 
import random
import re

import modp as mp

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import cartopy.crs as ccrs
import cartopy.feature as cf
from matplotlib import cm

/global/homes/j/jneedham/.conda/envs/myenv/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
random.seed(32)

#### Define new parameter ranges

In [3]:
param_ranges_full = pd.read_csv('/global/homes/j/jneedham/FATES-MRV/param_files/v3/PA_params_v3.csv')
param_ranges = param_ranges_full[['param', 'value_min', 'value_max', 'pft', 'organ']]

# number of parameters
n_params = len(param_ranges)

# number of PFTs - some are global so subtract one
n_pfts = len(pd.unique(param_ranges['pft'])) - 1

param_names = list(param_ranges.param)
pfts = list(param_ranges.pft)
organs = list(param_ranges.organ)

print(param_ranges)

                                 param  value_min  value_max  pft  organ
0                fates_leaf_vcmax25top  54.255559  60.886794    2    NaN
1                fates_leaf_vcmax25top  60.296702  67.666299    1    NaN
2                     fates_mort_bmort   0.013173   0.015967    2    NaN
3                     fates_mort_bmort   0.013469   0.016325    1    NaN
4  fates_recruit_seed_germination_rate   0.368751   0.516251    2    NaN
5  fates_recruit_seed_germination_rate   0.507542   0.710558    1    NaN


In [4]:
n_inst = 500

sampler = qmc.LatinHypercube(d=n_params)
sample = sampler.random(n=n_inst)

# scale to parameter ranges
l_bounds = param_ranges['value_min']
u_bounds = param_ranges['value_max']

scaled_sample = qmc.scale(sample, l_bounds, u_bounds)

In [5]:
npft1 = param_ranges['pft'].value_counts().get(1, 0)

print(npft1)

3


In [6]:
rows, cols = n_inst, npft1

df = pd.DataFrame(np.nan, index=range(rows), columns=range(cols))

## Read in defaut FATES file - note that this is the default for FATES but with:
#  two stream radiation
# Atkin respiration
# Updated allometries for Eastern US trees 

input_fname = '/global/homes/j/jneedham/FATES-MRV/param_files/v3/fates_params_2pfts_pa_ens_v2_238.nc'


# for each sample 
for i in range(0,n_inst) :
    
    # final parameter file name
    fout = '/global/homes/j/jneedham/FATES-MRV/param_files/v3/fates_params_2pfts_pa_ens_v3_{0}.nc'.format(i+1)
    
    shutil.copyfile(input_fname, fout)   
    
    pft1_ind = 0

   
    # loop through each parameter and apply either to the correct pft or globally
    for j in range(0, n_params) : 
        
        pft = pfts[j]
        organ = organs[j]
        val = scaled_sample[i, j]
        var = param_names[j]
        
        if pft == 1:
            
            # Parameters where PFT 1 val should be higher 
            if var in ['fates_mort_bmort', 
                       'fates_recruit_seed_germination_rate',
                       'fates_leaf_vcmax25top'] : 
            
                minv = scaled_sample[i, j-1]
                maxv = param_ranges['value_max'][j]
            
            # remaining PFT level parameters have higher value for PFT 2
            else:
                
                maxv = scaled_sample[i, j-1]
                minv = param_ranges['value_min'][j]
            
            pft1val = random.uniform(minv, maxv)
            mp.main(var = var, pft = pft, fin = fout, val = pft1val, 
                    fout = fout, O = 1, organ = organ)
            
            df.iloc[i, pft1_ind] = pft1val
            
            pft1_ind = pft1_ind + 1
        
        
        else:
            mp.main(var = var, pft = pft, fin = fout, val = val, 
                        fout = fout, O = 1, organ = organ)